# Before/After: fine-tune a small model on a Nemotron Data Forge dataset

This notebook fine-tunes a small instruct model on a dataset produced by the pipeline and compares answers **before** (base model) vs **after** (fine-tuned), with an offline quality report.

**No API keys are required.** For speed, choose **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# Install the fine-tuning dependencies (skip if already present).
!pip install -q "transformers>=4.40" "peft>=0.10" "datasets>=2.18" "accelerate>=0.30"

In [ ]:
import os
import subprocess
import sys

# Clone the project (or set WORKDIR to an existing checkout).
REPO = "https://github.com/Mxhesh-07/nemotron-data-forge.git"
WORKDIR = "nemotron-data-forge"
if not os.path.isdir(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
os.chdir(WORKDIR)
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

## Pick the dataset

Uses the newest pipeline output (`outputs/dataset-*.jsonl`) if present, otherwise the bundled Python-fundamentals fixture so the notebook still runs.

In [ ]:
import glob

DATASET = "finetune/sample_dataset.jsonl"
generated = sorted(glob.glob("outputs/dataset-*.jsonl"))
if generated:
    DATASET = generated[-1]
print("Using dataset:", DATASET)
with open(DATASET, encoding="utf-8") as fh:
    print("First line:", fh.readline().strip()[:200])

In [ ]:
# Validate the dataset and print the train/eval split (no GPU needed).
!python -m finetune.train_lora --dataset "$DATASET" --check

## Train + compare

Trains a LoRA adapter, then generates eval answers before/after and writes the report.

In [ ]:
!python -m finetune.train_lora \
  --dataset "$DATASET" \
  --base-model Qwen/Qwen2.5-0.5B-Instruct \
  --epochs 3 --eval-ratio 0.25 --eval-examples 8 \
  --out finetune_output

In [ ]:
from IPython.display import Markdown, display

display(Markdown(open("finetune_output/before_after.md", encoding="utf-8").read()))

In [ ]:
# Optional: download the report.
try:
    from google.colab import files
    files.download("finetune_output/before_after.md")
except Exception as exc:
    print("Not running in Colab, skipping download:", exc)